# Day 17 — Joins and Modelling: Primary and Foreign Keys, One-to-Many, Many-to-Many, INNER and LEFT JOIN & Normalisation (10 Nov)

*Rebuilt from the master Google Doc session (lesson text and every code example). Code cells are not executed: several need Postgres, a running server or files from the original starter zip.*

## Today at a glance

- Goal: stop storing everything in one wide table. Split data into linked tables (normalisation), link them with primary and foreign keys, and put them back together in queries with INNER JOIN, LEFT JOIN and CROSS JOIN, for one-to-many and many-to-many relationships.

- Deliverable: day17/schema.sql (Acme Retail's content model as six linked tables: authors, locales, content entries, their locale versions, tags and the entry-tag links) and day17/joins.sql (eight join questions J1 to J8 with exact expected outputs), loaded with the given day17/seed.sql. Plus day17/dsa.py.

- Where this sits: Week 3, Phase 1 (Foundation), day 3 of 7. Days 15 and 16 worked on one table at a time. From today you work the way every real system does: many tables, joined at query time. Day 18 loads data into these tables from Python, Day 19 adds constraints and indexes, and Week 6's RAG app joins document chunks to their source documents the same way.

Time split (5h): the Week 3 split, with the catch-up hour kept.

| Block | Time | What you do |
|---|---|---|
| Theory | 1.5h | Concepts 1 to 6, typing and running every example in psql |
| SQL hands-on | 2h | schema.sql (six tables), then J1 to J8 in joins.sql |
| Catch-up | 1h | Day 16: report.sql, R1 to R9 |
| DSA | 0.5h | hash_join and find_orphans in dsa.py |

Open items first: GitHub main still has two commits (Days 1 to 3, and Day 4's starters), and the Day 15 and 16 starters were sent as patches because the push was blocked. Apply them in order (git am on each file) so day15/ and day16/ exist on your laptop. Today's folder ships its own data, so it does not depend on them. Push at the end of the day even if something is unfinished.

Before you start: Postgres running and the fde database from Day 15. From inside day17/, run psql -d fde -v ON_ERROR_STOP=1 -f setup.sql once. It creates the lesson tables used in every example below: writers (4 rows), articles (6), topics (4), article_topics (6 links), languages (2), article_translations (9) and articles_flat, the same data as one wide table. Every output shown here was produced by running the query on PostgreSQL 16.

## Recap (2 minutes, from memory)

Three questions on Day 16. Answer out loud first; the answers are on the last page.

- pages has 8 rows and 2 of them have NULL views. What do count(*), count(views) and avg(views) each count or average over?

- You need "authors with more than one draft". Which part of the condition goes in WHERE, which goes in HAVING, and why can they not swap?

- WHERE locale NOT IN ('en-us', NULL) returns zero rows. Why, in one sentence?

## Concept 1: Normalisation: why one big table breaks

### Why it matters for an FDE

Customer data almost always arrives as one wide export: a CSV from a CMS, a CRM report, a spreadsheet someone has kept for years. The author's name and email are repeated on every row, tags sit in one cell separated by commas, and the same fact is typed slightly differently in different places. Before you can answer questions reliably, or give an AI assistant a database it can query without lying, you split that export into tables where each fact is stored once. This is the first design decision on almost every data project you will scope.

### How it works

Normalisation means arranging data so that every fact lives in exactly one place, in the table about the thing that fact describes. A writer's email is a fact about the writer, so it belongs in a writers table, stored once. An article only stores which writer it has, as a number pointing at that row. Look at the wide version first and find the problems:

```sql
-- Concept 1: the flat export repeats facts and hides lists in text.
\pset null '(null)'
SELECT uid, writer_name, writer_email, topics
FROM articles_flat
ORDER BY uid;
-- ->   uid   | writer_name |  writer_email  |    topics
-- -> --------+-------------+----------------+--------------
-- ->  blt201 | Asha        | asha@acme.in   | festive,sale
-- ->  blt202 | Ravi        | ravi@acme.in   | help
-- ->  blt203 | Ravi        | ravi.k@acme.in | help
-- ->  blt204 | Asha        | asha@acme.in   | sale
-- ->  blt205 | (null)      | (null)         | (null)
-- ->  blt206 | Meera       | meera@acme.in  | help
-- GOTCHA: Ravi has two emails. Which one is right? The table cannot say.
```

That one export shows the three classic problems, called anomalies:

- Update anomaly: a fact stored many times can be changed in one place and not the others. Ravi's email was fixed on one row only, so the data now contradicts itself.

- Insert anomaly: you cannot store a fact without an unrelated one. Kiran has joined the team but has written nothing yet, and there is no row to put her email in.

- Delete anomaly: deleting one fact deletes another. Delete the Size chart article and Meera's email is gone from the database.

The comma-separated topics column has its own problem: you can only search it with text matching, which is slow and wrong as soon as one topic name contains another.

```sql
-- Concept 1: searching a comma list is a text search, and text search lies.
SELECT uid, title FROM articles_flat WHERE topics LIKE '%sale%' ORDER BY uid;
-- ->   uid   |       title
-- -> --------+-------------------
-- ->  blt201 | Diwali gift guide
-- ->  blt204 | Monsoon sale
-- Correct today. Add a topic called 'presale' and it matches too.
-- No index can help either: '%...' at the start forces a full scan.
-- REMEMBER: one value per cell. A list in a cell is a missing table.
```

The textbook normal forms say the same thing in formal language. In plain words: first normal form means one value per cell (no lists in a column). Second and third normal form mean every non-key column describes the whole key of its own table and nothing else (a writer's email does not belong in the articles table, because it describes the writer, not the article). If you remember "each fact once, in the table about that thing", you have the useful part.

You already know this from the frontend. A normalised Redux store keeps users by id and has posts hold a userId, instead of copying the user object into every post. In Contentstack it is a reference field: an entry stores the uid of the author entry, not a copy of the author's name and email, so editing the author once updates every page that shows them.

### Real-world FDE example

A retailer's "product master" spreadsheet has brand name, brand logo URL and brand contact on all 40,000 product rows. Their chatbot answers "who do I contact about Brand X" with three different emails, because the rows disagree. In the proposal you show the SELECT DISTINCT brand, brand_email result (three rows for one brand), recommend a brands table with one row per brand, and scope a one-time cleanup where the customer picks the correct email for each conflict. Normalisation turns a vague "the bot is inconsistent" complaint into a specific, fixable data task.

### Common mistakes

- Storing lists in a column (comma-separated tags, JSON arrays of ids) because the export looked that way. Make a table instead. Postgres does have array and JSONB types; Day 20 shows when they are the right tool.

- Splitting too far. Not every column needs a table: a page's title is a fact about that page and stays in its row. Split when a group of columns describes a different thing, or when one row needs several values.

- Fixing duplicates by hand in the wide table. The duplicates come back with the next export. Change the structure so the duplicate cannot be stored.

## Concept 2: Primary keys and foreign keys

### Why it matters for an FDE

Keys are how tables point at each other. A primary key gives every row an identity that never changes; a foreign key is a column that holds another table's primary key, and the database guarantees it points at a row that exists. When you integrate with a customer's system, the first question is always "what is the id of this thing, and is it stable?". Get it wrong and a sync job duplicates records or links a ticket to the wrong customer.

### How it works

A primary key (PK) is a column, or a set of columns, that is unique and never NULL. Postgres builds an index on it automatically. There are two kinds. A surrogate key is a number the database makes up, such as id integer GENERATED ALWAYS AS IDENTITY, which Postgres fills with 1, 2, 3 for you (like an auto-increment id). A natural key is a real-world value that is already unique and stable, such as a locale code 'en-us'. Use surrogate ids for things whose names can change (people, entries), and natural keys only for short codes that never change.

A foreign key (FK) is declared with REFERENCES. It is the database version of a React key lookup or a Contentstack reference field, except the database enforces it: you cannot store a writer id that does not exist, and you cannot delete a writer while articles still point at them.

```sql
-- Concept 2: the keys in setup.sql (shortened). Parents first.
CREATE TABLE writers (
    id    integer GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    name  text NOT NULL,
    email text NOT NULL UNIQUE        -- stored once, so it cannot disagree
);

CREATE TABLE articles (
    id        integer GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    uid       text NOT NULL UNIQUE,   -- the CMS id, kept for syncing
    title     text NOT NULL,
    status    text NOT NULL,
    -- FK: must match a writers.id, or be NULL (= unassigned)
    writer_id integer REFERENCES writers (id)
);
-- FDE TIP: keep the source system's id (uid) as its own UNIQUE column.
-- Your own id is for joins; the uid is how the next sync finds the row.
```

Now watch the foreign key protect the data. Both errors below are the database refusing to let the tables contradict each other. Read the DETAIL line: it names the exact key.

```sql
-- Concept 2: a foreign key blocks orphans in both directions.
INSERT INTO articles (uid, title, status, writer_id)
VALUES ('blt299', 'Test', 'draft', 99);
-- -> ERROR:  insert or update on table "articles" violates foreign key
-- ->         constraint "articles_writer_id_fkey"
-- -> DETAIL:  Key (writer_id)=(99) is not present in table "writers".

DELETE FROM writers WHERE id = 3;     -- Meera still has the Size chart
-- -> ERROR:  update or delete on table "writers" violates foreign key
-- ->         constraint "articles_writer_id_fkey" on table "articles"
-- -> DETAIL:  Key (id)=(3) is still referenced from table "articles".
-- REMEMBER: the FK lives on the "many" side and points at the "one" side.
```

What should happen to the children when a parent row is deleted? You choose with ON DELETE. The default (no action, like RESTRICT) refuses, as above. ON DELETE CASCADE deletes the children too, which is right for rows that mean nothing without their parent, such as an article's topic links. ON DELETE SET NULL keeps the child and clears the link, which suits "the author left, keep the article".

```sql
-- Concept 2: CASCADE removes the link rows along with their article.
-- BEGIN ... ROLLBACK runs a change and then undoes it (Day 18 covers it).
BEGIN;
DELETE FROM articles WHERE uid = 'blt201';  -- the Diwali gift guide
SELECT count(*) AS links_left FROM article_topics;
-- ->  links_left
-- -> ------------
-- ->           4     (was 6: its festive and sale links went with it)
ROLLBACK;
SELECT count(*) AS links_back FROM article_topics;
-- ->  links_back
-- -> ------------
-- ->           6
-- GOTCHA: CASCADE on the wrong FK (writers -> articles) would let one
-- DELETE of a writer wipe out all their published pages.
```

### Real-world FDE example

A customer's nightly job copied Salesforce accounts into Postgres keyed by account name. When "Globex Ltd" was renamed "Globex Group", the job created a second account and the old one kept all the open opportunities. The fix you propose: key on Salesforce's stable record id (a natural key from the source, stored as a UNIQUE column), use an internal surrogate id for joins, and add a foreign key from opportunities to accounts so an opportunity can never point at an account that does not exist.

### Common mistakes

- Using a name or email as the primary key. Names change, and every table that points at the old value then breaks. Use a surrogate id and keep the name as a normal column.

- Creating or dropping tables in the wrong order. A table must exist before another can reference it, and a parent cannot be dropped while children point at it. Create parents first and drop children first (or list all of them in one DROP TABLE).

- Giving the FK column a different type from the key it points at (text against integer). Postgres refuses to create it. Copy the parent key's exact type.

- Adding ON DELETE CASCADE everywhere "so deletes work". Decide per relationship whether the child can live without its parent.

## Concept 3: One-to-many and INNER JOIN

### Why it matters for an FDE

Once data is split, every useful answer needs it put back together: an article with its writer's name, a ticket with its customer's plan, a chunk of text with the title of the document it came from. JOIN does that inside the database. It is also where most wrong numbers come from, so you need to know exactly which rows a join keeps.

### How it works

A one-to-many relationship is the most common shape: one writer has many articles, and each article has at most one writer. It is stored as a foreign key on the many side (articles.writer_id). An inner join pairs every row of the left table with every row of the right table where the ON condition is true, and keeps only those pairs. A row with no partner disappears from the result.

In JavaScript terms: articles.flatMap(a => writers.filter(w => w.id === a.writer_id).map(w => ({ ...a, writer: w.name }))). The database does the same thing, but with indexes and without loading everything into memory.

```sql
-- Concept 3: each article next to its writer's name.
SELECT a.uid, a.title, w.name AS writer   -- a and w are table aliases
FROM articles AS a
JOIN writers  AS w ON w.id = a.writer_id  -- JOIN means INNER JOIN
ORDER BY a.uid;
-- ->   uid   |       title       | writer
-- -> --------+-------------------+--------
-- ->  blt201 | Diwali gift guide | Asha
-- ->  blt202 | Return policy     | Ravi
-- ->  blt203 | Store locator     | Ravi
-- ->  blt204 | Monsoon sale      | Asha
-- ->  blt206 | Size chart        | Meera
-- 5 rows, but articles has 6. blt205 (Careers) has writer_id NULL, and
-- NULL = anything is unknown, so it has no partner and is dropped.
-- Kiran has no articles, so she never appears either.
```

Aliases (articles AS a) keep queries short, and you need them as soon as two tables share a column name. Both tables here have an id column, so an unqualified id is an error. Always write table_alias.column in a join, even when it is not ambiguous; the next person (or your future AI agent reading the query) can then see where each column comes from.

```sql
-- Concept 3: unqualified names fail when both tables have the column.
SELECT uid, title, name
FROM articles JOIN writers ON writers.id = articles.writer_id
WHERE id = 1;
-- -> ERROR:  column reference "id" is ambiguous
-- Fix: WHERE articles.id = 1 (or use aliases: a.id = 1).

-- Filtering on the joined table: Ravi's articles, by name not by id.
SELECT a.uid, a.title
FROM articles AS a
JOIN writers  AS w ON w.id = a.writer_id
WHERE w.name = 'Ravi'
ORDER BY a.uid;
-- ->   uid   |     title
-- -> --------+---------------
-- ->  blt202 | Return policy
-- ->  blt203 | Store locator
```

Where does the join sit in the order of work from Day 16? It is part of FROM, the very first step. The database builds the joined rows first, then WHERE filters them, then GROUP BY, HAVING, SELECT, ORDER BY and LIMIT run exactly as before. So everything you learned on Days 15 and 16 works on a join result as if it were one table.

### Real-world FDE example

A support team wants "open tickets with the customer's plan and account manager". tickets has customer_id, customers has plan and manager_id, managers has the name. Two joins give one row per ticket with all three facts. Before you show the dashboard, you compare SELECT count(*) FROM tickets WHERE status = 'open' with the count from the join: if the join returns fewer rows, some tickets point at customers that are missing from the sync, and that gap goes into your status report.

### Common mistakes

- Forgetting that an inner join drops rows with no partner. Always compare the join's row count with the main table's count. A smaller number means missing or NULL links.

- Joining on the wrong columns, for example ON w.id = a.id. It runs without an error and pairs article 1 with writer 1. Join a foreign key to the primary key it references.

- Writing the old comma syntax, FROM articles, writers, and forgetting the WHERE. That pairs every article with every writer (6 x 4 = 24 rows) and every total is wrong. Use explicit JOIN ... ON.

## Concept 4: LEFT JOIN: keep every row, find what is missing

### Why it matters for an FDE

Many of the most useful questions are about absence: writers with no articles, customers with no orders, documents that were never indexed, products with no price. An inner join cannot answer them, because it drops exactly the rows you are looking for. LEFT JOIN keeps them.

### How it works

A left join keeps every row of the left table (the one written first). Where a row has a partner, you get the pair as usual. Where it has none, you still get the left row, with NULL in every column of the right table. It works like optional chaining in JavaScript: article.writer?.name gives undefined instead of dropping the article.

```sql
-- Concept 4: every article, with or without a writer.
\pset null '(null)'
SELECT a.uid, a.title, w.name AS writer
FROM articles AS a
LEFT JOIN writers AS w ON w.id = a.writer_id
ORDER BY a.uid;
-- ->   uid   |       title       | writer
-- -> --------+-------------------+--------
-- ->  blt201 | Diwali gift guide | Asha
-- ->  blt202 | Return policy     | Ravi
-- ->  blt203 | Store locator     | Ravi
-- ->  blt204 | Monsoon sale      | Asha
-- ->  blt205 | Careers           | (null)   <- kept, writer columns NULL
-- ->  blt206 | Size chart        | Meera
-- FDE TIP: show it as coalesce(w.name, 'Unassigned') for humans (Day 16).
```

Put the table you want to keep in full on the left. To count articles per writer including writers with none, writers goes first. Then count a column from the right table, not *: the writer with no articles still has one result row (full of NULLs), so count(*) says 1, while count(a.id) skips the NULL and says 0.

```sql
-- Concept 4: articles per writer, zeros included.
SELECT w.name,
       count(a.id) AS articles,    -- counts real partners only
       count(*)    AS rows_seen    -- counts result rows, NULL-filled too
FROM writers AS w
LEFT JOIN articles AS a ON a.writer_id = w.id
GROUP BY w.id, w.name              -- group by the key, not just the name
ORDER BY articles DESC, w.name;
-- ->  name  | articles | rows_seen
-- -> -------+----------+-----------
-- ->  Asha  |        2 |         2
-- ->  Ravi  |        2 |         2
-- ->  Meera |        1 |         1
-- ->  Kiran |        0 |         1     <- count(*) would report 1 article

-- The anti-join: keep only the rows that found NO partner.
SELECT w.name
FROM writers AS w
LEFT JOIN articles AS a ON a.writer_id = w.id
WHERE a.id IS NULL;                -- NULL here means "no match"
-- ->  name
-- -> -------
-- ->  Kiran
-- REMEMBER: LEFT JOIN + WHERE right.id IS NULL finds what is missing.
```

The trap that catches everyone: a condition on the right table in WHERE undoes the left join. WHERE runs after the join, and for a row with no partner the right-hand columns are NULL, so a.status = 'published' is unknown and the row is thrown away. Kiran disappears again. To filter the right table and still keep every left row, put the condition in ON, so it decides which partners match, not which rows survive.

```sql
-- Concept 4: the same filter in WHERE versus in ON.
SELECT w.name, a.uid
FROM writers AS w
LEFT JOIN articles AS a ON a.writer_id = w.id
WHERE a.status = 'published'       -- runs AFTER the join
ORDER BY w.name, a.uid;
-- ->  Asha | blt201,  Asha | blt204,  Meera | blt206,  Ravi | blt202
-- -> (4 rows)   Kiran is gone: it behaves like an INNER JOIN now

SELECT w.name, a.uid
FROM writers AS w
LEFT JOIN articles AS a
       ON a.writer_id = w.id
      AND a.status = 'published'   -- part of the match rule
ORDER BY w.name, a.uid;
-- ->  Asha | blt201,  Asha | blt204,  Kiran | (null),
-- ->  Meera | blt206,  Ravi | blt202
-- -> (5 rows)   every writer kept; only published articles attached
-- GOTCHA: right-table filters go in ON when you need the zeros.
```

Two relatives you will meet in other people's SQL: RIGHT JOIN is a left join written the other way round (keep every row of the second table), and most teams avoid it by swapping the table order. FULL JOIN keeps unmatched rows from both sides, which is useful for reconciling two systems: rows only in the CMS, rows only in the search index, and rows in both.

### Real-world FDE example

A RAG deployment answers "I don't know" for some products. You join the products table to the embedded chunks with a left join and keep the rows where the chunk side is NULL: 312 of 4,800 products were never embedded because their descriptions were empty when the indexing job ran. That one anti-join query is the root cause, the fix (re-run indexing for those ids) and the monitoring check (alert when the count is above zero) for the runbook.

### Common mistakes

- Putting a right-table condition in WHERE after a LEFT JOIN, which silently turns it into an inner join. Move it into ON.

- Counting count(*) after a left join and reporting 1 for things that have none. Count a right-table column, usually its primary key.

- Putting the tables in the wrong order. FROM articles LEFT JOIN writers keeps every article; FROM writers LEFT JOIN articles keeps every writer. Decide which list must be complete.

## Concept 5: Many-to-many: junction tables and the fan-out trap

### Why it matters for an FDE

Tags on entries, users in groups, products in categories, documents with permissions for many teams: many-to-many relationships are everywhere in enterprise systems, and permissions in particular decide what an AI assistant is allowed to show each user. You need to model them correctly and, just as important, aggregate across them without inflating every number.

### How it works

One article has many topics and one topic is on many articles. A foreign key column cannot hold that, because one cell would need several values (the comma list from Concept 1). The answer is a junction table (also called a link or join table) with one row per pair: article_topics(article_id, topic_id). Its primary key is both columns together, a composite key, so the same pair cannot be stored twice. Each column is also a foreign key. In Contentstack terms it is a multi-reference field, stored as rows instead of an array.

```sql
-- Concept 5: the junction table from setup.sql.
CREATE TABLE article_topics (
    article_id integer REFERENCES articles (id) ON DELETE CASCADE,
    topic_id   integer REFERENCES topics (id)   ON DELETE CASCADE,
    PRIMARY KEY (article_id, topic_id)    -- each pair at most once
);

-- Two joins: article -> link rows -> topic. One result row per link.
SELECT a.title, t.name AS topic
FROM articles       AS a
JOIN article_topics AS at ON at.article_id = a.id
JOIN topics         AS t  ON t.id = at.topic_id
ORDER BY a.title, t.name;
-- ->        title       |  topic
-- -> -------------------+---------
-- ->  Diwali gift guide | festive
-- ->  Diwali gift guide | sale
-- ->  Monsoon sale      | sale
-- ->  Return policy     | help
-- ->  Size chart        | help
-- ->  Store locator     | help
-- Careers has no topics, so the inner joins drop it.
```

To show one row per article with its topics as a list, group by the article and use string_agg(value, separator ORDER BY ...), an aggregate that joins text the way array.join(', ') does in JavaScript. Left joins keep the article with no topics. The same pattern from the other side counts how often each topic is used, zeros included.

```sql
-- Concept 5: topics as one string per article, and topic usage.
\pset null '(null)'
SELECT a.title,
       string_agg(t.name, ', ' ORDER BY t.name) AS topics
FROM articles AS a
LEFT JOIN article_topics AS at ON at.article_id = a.id
LEFT JOIN topics         AS t  ON t.id = at.topic_id
GROUP BY a.id, a.title
ORDER BY a.title;
-- ->        title       |    topics
-- -> -------------------+---------------
-- ->  Careers           | (null)        <- no links: string_agg of nothing
-- ->  Diwali gift guide | festive, sale
-- ->  Monsoon sale      | sale
-- ->  Return policy     | help
-- ->  Size chart        | help
-- ->  Store locator     | help

SELECT t.name, count(at.article_id) AS articles
FROM topics AS t
LEFT JOIN article_topics AS at ON at.topic_id = t.id
GROUP BY t.id, t.name
ORDER BY articles DESC, t.name;
-- ->   name   | articles
-- -> ---------+----------
-- ->  help    |        3
-- ->  sale    |        2
-- ->  festive |        1
-- ->  brand   |        0     <- unused topic, kept by the LEFT JOIN
```

Now the fan-out trap. A join produces one row per matching pair, so when you join through a many-to-many link, a parent row appears once per link. Asha has two articles, but the Diwali gift guide has two topics, so after joining topics she appears three times. Any count(*) or sum over those rows counts her articles more than once. Nothing errors; the numbers are just wrong.

```sql
-- Concept 5: joining through topics multiplies the writer's rows.
SELECT w.name, count(*) AS articles_wrong
FROM writers        AS w
JOIN articles       AS a  ON a.writer_id = w.id
JOIN article_topics AS at ON at.article_id = a.id
GROUP BY w.name
ORDER BY w.name;
-- ->  name  | articles_wrong
-- -> -------+----------------
-- ->  Asha  |              3     <- she has 2 articles
-- ->  Meera |              1
-- ->  Ravi  |              2

-- Fix 1: count each article once.
SELECT w.name, count(DISTINCT a.id) AS articles
FROM writers        AS w
JOIN articles       AS a  ON a.writer_id = w.id
JOIN article_topics AS at ON at.article_id = a.id
GROUP BY w.name
ORDER BY w.name;
-- ->  Asha 2,  Meera 1,  Ravi 2
-- Fix 2 (better): do not join tables the question does not need.
-- GOTCHA: count(DISTINCT) fixes counts, but sum(DISTINCT views) would
-- also merge two different articles that happen to have equal views.
```

### Real-world FDE example

A finance dashboard showed revenue per region 40 percent higher than the ledger. The query joined orders to order_tags (a many-to-many link for campaign tags) to filter "campaign orders", and then summed orders.amount. Every order with two tags was counted twice. The fix was to filter with WHERE EXISTS (...) on the tag table instead of joining it (Concept 6 shows EXISTS), so each order stays one row. Checking a total against a known number from the source system is the habit that catches this.

### Common mistakes

- Storing a many-to-many relationship as a comma list or as two foreign key columns (topic1_id, topic2_id). Use a junction table.

- Leaving the composite primary key off the junction table, so the same pair can be inserted twice and every count doubles.

- Summing a parent's column after joining a child or link table. Join only what the question needs, or aggregate before you join.

- Grouping by name alone. Two writers called Ravi would merge into one group. Group by the primary key and the name.

## Concept 6: Modelling a headless CMS: locales and finding the gaps

### Why it matters for an FDE

Your edge is that you know how headless CMSs model content. When a customer asks for "an AI assistant over our website content" or "a translation dashboard", you will design the tables that hold their entries, and the most important CMS-specific decision is how to store one entry in many languages. Get it right and "which pages are not translated into Marathi?" is one query.

### How it works

A four-step design recipe that works for most models:

- List the nouns in the customer's language: entry, author, locale, tag. Each one becomes a table with a primary key.

- For each pair of nouns, ask "one or many?" in both directions. One author, many entries: a foreign key on entries. Many entries, many tags: a junction table.

- Find facts that depend on a combination. An entry's title and status differ per locale, so they belong to the pair (entry, locale): a child table with a composite primary key. The content type does not change per locale, so it stays on the entry.

- Keep the source system's ids (the Contentstack uid) as unique columns, so a sync can find each row again.

This is the same split Contentstack makes: an entry has one uid and one content type, and each locale version has its own title, fields and publish status. In the lesson tables, article_translations holds one row per (article, language) that exists.

To find what is missing, first build every combination that should exist, then left join to what does exist. CROSS JOIN makes every pairing of two tables with no condition: 6 articles x 2 languages = 12 pairs. It is the same as two nested loops in JavaScript. The anti-join from Concept 4 then keeps the pairs with no translation row.

```sql
-- Concept 6: every (article, language) pair that SHOULD exist.
SELECT count(*) AS pairs FROM articles CROSS JOIN languages;
-- ->  pairs
-- -> -------
-- ->     12      (6 articles x 2 languages, no ON condition)

-- Keep the pairs with no translation: the translation backlog.
SELECT a.uid, l.code AS missing
FROM articles  AS a
CROSS JOIN languages AS l
LEFT JOIN article_translations AS tr
       ON tr.article_id = a.id
      AND tr.lang_code  = l.code      -- match on BOTH key columns
WHERE tr.article_id IS NULL
ORDER BY a.uid, l.code;
-- ->   uid   | missing
-- -> --------+---------
-- ->  blt203 | hi-in
-- ->  blt204 | hi-in
-- ->  blt205 | hi-in
-- GOTCHA: CROSS JOIN on big tables explodes (10,000 x 10,000 = 100M rows).
-- It is safe here because one side is a short list of locales.
```

The same question for one locale can be written with NOT EXISTS, which reads closer to English: keep the article if there is no translation row for it. EXISTS (subquery) is true as soon as the inner query finds one row, so it never multiplies rows, which is also why it fixes the fan-out trap when you only need to filter. Prefer NOT EXISTS over NOT IN with a subquery: Day 16 showed that one NULL inside a NOT IN list returns nothing.

```sql
-- Concept 6: the same gap for Hindi, written with NOT EXISTS.
SELECT a.uid
FROM articles AS a
WHERE NOT EXISTS (
    SELECT 1                           -- the value does not matter
    FROM article_translations AS tr
    WHERE tr.article_id = a.id         -- linked to the outer row
      AND tr.lang_code  = 'hi-in'
)
ORDER BY a.uid;
-- ->  blt203, blt204, blt205    (3 rows, the same answer)

-- The composite key refuses a second English version of article 1.
INSERT INTO article_translations VALUES (1, 'en-us', 'Duplicate');
-- -> ERROR:  duplicate key value violates unique constraint
-- ->         "article_translations_pkey"
-- -> DETAIL:  Key (article_id, lang_code)=(1, en-us) already exists.
```

### Real-world FDE example

Acme Retail wants its Marathi site live before a regional campaign. You load their Contentstack export into entries and entry_locales, then the cross join plus anti-join lists every (entry, locale) gap, grouped by content type and owner. The product team sees 43 untranslated product pages owned by two people. That list becomes the translation plan, and the same query, run nightly, becomes a progress tile on the dashboard you build in Week 4.

### Common mistakes

- One column per locale (title_en, title_hi, title_mr). Adding a locale then needs a schema change. Use one row per (entry, locale).

- Putting per-locale facts on the entry (status on content_entries) or shared facts on the locale table (content type repeated per locale). Ask: does this value change when the language changes?

- Matching the anti-join on only one key column (ON tr.article_id = a.id). Any translation in any language then counts as a match, and no gaps are found.

## Check your understanding (predict first, then run)

Write a prediction for each query in the notebook, then run it against the lesson tables. If a result surprises you, find which row the join kept or dropped.

```sql
-- Check your understanding: predict each result, then run it.
\pset null '(null)'
-- 1. How many rows?
SELECT a.uid, w.name FROM articles a JOIN writers w ON w.id = a.writer_id;
-- 2. How many rows, and which writer appears with a NULL?
SELECT w.name, a.uid FROM writers w LEFT JOIN articles a
    ON a.writer_id = w.id;
-- 3. What number comes back?
SELECT count(*) FROM writers CROSS JOIN topics;
-- 4. Which topic names, and why not 'brand'?
SELECT DISTINCT t.name FROM topics t
JOIN article_topics at ON at.topic_id = t.id ORDER BY t.name;
-- 5. Two numbers. Why are they different?
SELECT count(*), count(DISTINCT a.id)
FROM articles a JOIN article_topics at ON at.article_id = a.id;
-- 6. How many rows? (Hint: where is the status condition?)
SELECT w.name, count(a.id) FROM writers w
LEFT JOIN articles a ON a.writer_id = w.id
WHERE a.status = 'draft' GROUP BY w.id, w.name;
```

## Hands-on task (about 2 hours): day17/schema.sql and day17/joins.sql

The scenario: Acme Retail's Contentstack export has been one wide table so far (Days 15 and 16). Their content lead wants answers that table cannot give reliably: who owns what, which tags are used, and which pages are missing in Hindi and Marathi. You design the normalised model and answer eight questions with joins. The starter files have the brief, the exact table spec, the expected output for each question and hints above every TODO. Do not edit seed.sql; the expected outputs depend on its rows.

- Pull and load the lesson tables (10 min). git pull, then from inside day17/ run psql -d fde -v ON_ERROR_STOP=1 -f setup.sql if you have not already.

- Write schema.sql (40 min). The brief at the top lists six tables: authors, locales, content_entries, entry_locales, tags and entry_tags, with every column, type, key and ON DELETE rule. The parent table is called content_entries so it does not clash with Day 15's entries table in the same database. Start the file with one DROP TABLE IF EXISTS that lists all six, so it can be re-run. Then answer the three design questions D1 to D3 in the comments.

- Load the data (5 min). psql -d fde -v ON_ERROR_STOP=1 -f schema.sql -f seed.sql. If seed.sql fails, the error names the table or column that does not match the brief. Fix schema.sql and run both again. Check with \d entry_locales: two foreign keys and a two-column primary key.

- Answer J1 to J8 (60 min). One query per question under its TODO in joins.sql. Run each alone in psql, compare it with its Expected comment, then save it.

- Run the whole file (5 min). psql -d fde -v ON_ERROR_STOP=1 -f joins.sql. Every result must match.

- Commit and push. git add day17 && git commit -m "Day 17: normalised CMS schema and join queries" && git push.

The eight questions (full wording and expected outputs are in the file):

| # | Question | Main idea |
|---|---|---|
| J1 | Entries with their author's name (owned entries only) | INNER JOIN, one-to-many |
| J2 | Every entry with its author, Unassigned when none | LEFT JOIN, coalesce |
| J3 | Entries per author, including authors with none | LEFT JOIN, count(col) |
| J4 | Published versions and views per locale, by locale name | join a lookup table, GROUP BY |
| J5 | Every entry's English title and its tags as one string | two joins through a junction, string_agg |
| J6 | Tag usage, including unused tags | LEFT JOIN from the many-to-many side |
| J7 | Missing translations: every (entry, locale) with no version | CROSS JOIN plus anti-join |
| J8 | Published views per author: fix a given wrong query | the fan-out trap |

Acceptance criteria:

- psql -d fde -v ON_ERROR_STOP=1 -f schema.sql -f seed.sql runs with no ERROR lines, twice in a row.

- \d on each table shows the primary keys, foreign keys, UNIQUE columns and ON DELETE rules from the brief.

- Each of J1 to J8 matches its Expected comment exactly: columns, names, row count and every value.

- Every column in a join query is qualified with a table alias; every multi-row result has an ORDER BY with a tie-breaker; no SELECT *.

- D1 to D3 in schema.sql and the J8 explanation are answered in your own words, in one to three sentences each.

- The day17 folder is pushed to main.

Hints, if you are stuck (try for 10 minutes first):

- schema.sql: create parents before children (authors and locales, then content_entries, then the rest). The FK column's type must match the key it points at: integer for ids, text for the locale code.

- J3 and J6: which table must be complete? Put it on the left, and count a column of the right table.

- J4: join entry_locales to locales to get the name. The published filter is about rows, so it goes in WHERE.

- J5: start from content_entries, join entry_locales for the English title (a condition on both entry and locale), then left join entry_tags and tags. Wrap string_agg in coalesce for entries with no tags.

- J7: Concept 6's pattern, with your table names. Match the left join on both key columns.

- J8: run the given query, then remove one table from it. Ask which table the question does not need.

Catch-up lane (1h): Day 16. Finish report.sql from the Day 16 patch. Done when psql -d fde -v ON_ERROR_STOP=1 -f setup.sql -f report.sql runs with no ERROR and R1 to R9 match. Push it as its own commit before today's.

## DSA practice (30 min)

Two problems in day17/dsa.py that are today's SQL written by hand. Signatures, examples and hints are in the file; the expected outputs are in its __main__ block.

- hash_join(left, right, left_key, right_key). An inner join of two lists of dicts in O(n + m) time, which is how Postgres runs most joins (a "hash join"). Hint: loop over right once and build a dict from key to the list of rows with that key, then loop over left and look each key up. Two nested loops would also work but are O(n x m).

- find_orphans(children, parents, fk, pk). Return the children whose foreign key points at no parent: the check a foreign key does for you. Hint: put every parent key in a set (Day 3), then keep the children whose key is not None and not in the set.

## Quiz (3 quick questions)

- A customer's export has a categories column like "shoes,sale,new". Name the problem and the tables you would create instead, with their keys.

- Write the query: every locale's name with the number of entries that have a version in it, including locales with zero, most first. Use the task tables.

- A dashboard joins orders to order_items and shows sum(orders.shipping_fee) per day. The total is too high. Why, and what are two ways to fix it?

## What you achieved today

You learned why one wide table produces contradictions and how normalisation stores each fact once, used primary and foreign keys to link tables and let the database protect those links, joined one-to-many data with INNER and LEFT JOIN, modelled many-to-many with a junction table, avoided the fan-out trap, and found missing translations with a cross join and an anti-join. Your schema.sql is the first real data model in the repo, and it is the shape Acme's data will keep for the rest of Phase 1.

Tomorrow (Day 18): Python and Postgres: psycopg 3, parameterised queries and why they stop SQL injection, and transactions, to load Day 6's CMS export into these tables in one go.

## When you're done

Paste into the Forward Deployment Engineer project chat as "Day 17 review": your recap answers, your six predictions next to the real outputs, schema.sql with D1 to D3, joins.sql (or the commit link), your dsa.py, your quiz answers and the Day 16 catch-up status. Note anything that felt unclear; tomorrow starts there.

## Recap answers

*Your answers:*
